## 🧪 Experiments


In [ ]:
# Imports
from dotenv         import load_dotenv
from pathlib        import Path
import pandas       as pd
import datetime
import importlib.util
import os

# Custom Imports (run this notebook from its own directory)
import sys
notebookDir = Path.cwd().resolve()
sys.path.append(str(notebookDir.parent))
import AppUtils
import LLMUtils
from AnalysisUtils import AnalysisUtils, OBFUSCATION_TECHNIQUES, getClassificationLabelCount

# Share the APK analysis and result validation with the command-line driver.
driverSpec = importlib.util.spec_from_file_location("classification_experiments", notebookDir / "experiments.py")
classificationDriver = importlib.util.module_from_spec(driverSpec)
driverSpec.loader.exec_module(classificationDriver)


##### Initialization


In [ ]:
print("⚡ START: {} ⚡".format(datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")))
initTime = datetime.datetime.now()

In [ ]:
# Paths
TMP_PATH = "../../0_Data/TMP"

if not os.path.exists(TMP_PATH):
    os.makedirs(TMP_PATH)
    print("--- 🆕 Folder created : {}".format(TMP_PATH))
else:
    print("--- 📂 TMP Folder ready : {}".format(TMP_PATH))

In [ ]:
# GPU Test
import torch

# Check if CUDA is available
print("--- 🖥️ CUDA available           : {}".format(torch.cuda.is_available()))

# Show the number of GPUs available
numGpus = torch.cuda.device_count()
print("--- 🖥️ Number of available GPUs : {}".format(numGpus))

# Display the names of the GPUs
for i in range(numGpus):
	print("--- 🖥️ GPU {}: {}".format(i, torch.cuda.get_device_name(i)))

In [ ]:
# Load .env Info
load_dotenv()

#### 📥 1] Load Data


In [ ]:
# Where the input APKs are located
INPUT_APKS_PATH = os.path.abspath(os.path.join(os.getcwd(), "../../0_Data/InputApks/ClassRename"))

# Share filename parsing and APK metadata with the command-line driver.
# The folder is a group name; only single-technique runs use it as ground truth.
apkRecords, obfuscationTechniqueDirName = classificationDriver.loadApkRecords(Path(INPUT_APKS_PATH))
appsDF = pd.DataFrame(apkRecords)

if len(appsDF) == 0:
    raise ValueError("No APK files found in {}".format(INPUT_APKS_PATH))

# Show first 3 rows without limiting the dataset.
appsDF.head(3)


### 🧪 2] LLM Analysis


In [ ]:
# PAYING
# MODEL = "gpt-4o-mini"

# PAYING
# MODEL = "gemini-3-flash-preview"

# FREE
# MODEL = "gpt-oss:20b"
# MODEL = "deepseek-r1:32b"
# MODEL = "gemma3:27b"
MODEL = "qwen3:30b"
# MODEL = "llama3.1:8b"
# MODEL = "phi3:14b"

# Context Window Size
CONTEXT_WINDOW_SIZE = 128000
CONTEXT_THRESHOLD   = 0.8 * CONTEXT_WINDOW_SIZE

# The shared driver initializes the model after prompt validation and resume checks,
# and only when APKs remain to be analyzed.


##### Parameters


In [ ]:
# To compute the minimum number of classes for statistically significant random sample
CONFIDENCE_LEVEL    = 95
ERROR_MARGIN        = 5

# Canonical ten-technique label set shared with the classification driver.
EXPECTED_LABELS = list(OBFUSCATION_TECHNIQUES)

# Filtering [None | "system" | "tp" | "both" | "pkgNameOnly"]
FILTERING = "both"

# LLM Robustness
NUM_ITERATIONS  = 5
MAX_RETRIES     = 3

# Logging
SILENT_MODE = True


In [ ]:
# Prompt Selection: one label per class, or three labels per class for RQ5.
PROMPTS_PATH = "../prompt.yaml"
PROMPT_ID    = "ObfuscationClassificationV1"
# PROMPT_ID = "ObfuscationMultiClassificationV1"
NUM_LABELS = getClassificationLabelCount(PROMPT_ID)

prompts         = AnalysisUtils.loadPrompts(PROMPTS_PATH)
promptInfo      = AnalysisUtils.getPromptById(prompts, PROMPT_ID)
promptTemplate  = promptInfo["promptTemplate"]

print("--- 📝 Prompt ID          : {}".format(promptInfo["promptID"]))
print("--- 📝 Labels per Class   : {}".format(NUM_LABELS))
print("--- 📝 Prompt Description : {}".format(promptInfo["promptDescription"]))
print("--- 📝 Prompt Template    : {}".format(promptTemplate))


In [ ]:
# Results & Resume
RESULTS_DIR_PATH    = os.path.abspath(os.path.join(os.getcwd(), "../../0_Data/Results/", PROMPT_ID, obfuscationTechniqueDirName))
MODEL_FILE_NAME     = MODEL.replace(":", "_").replace("/", "_")
OUTPUT_PATH_JSON    = os.path.join(RESULTS_DIR_PATH, "results_{}.json".format(MODEL_FILE_NAME))
OUTPUT_PATH_CSV     = os.path.join(RESULTS_DIR_PATH, "results_{}.csv".format(MODEL_FILE_NAME))

# Reuse only validated results for this label count. The shared loader backs up
# the original JSON before discarding legacy, failed, or incompatible records.
classificationResults = classificationDriver.loadReusableClassificationResults(
    OUTPUT_PATH_JSON, NUM_LABELS, EXPECTED_LABELS
)
completedSha256Set = {result["sha256"] for result in classificationResults}
pendingAppsDF = appsDF[~appsDF["sha256"].isin(completedSha256Set)].reset_index(drop=True)

print("\n--- ⭕ Checking for existing results...")
print("--- 🔄️ Reusable Results : {}".format(len(classificationResults)))
print("--- 🔄️ Pending Apps     : {}/{}".format(len(pendingAppsDF), len(appsDF)))


In [ ]:
# Use the same analysis path as the CLI, with the notebook's parameters.
classificationDriver.TMP_PATH = Path(TMP_PATH).expanduser().resolve()
classificationDriver.FILTERING = FILTERING
classificationDriver.CONTEXT_WINDOW_SIZE = CONTEXT_WINDOW_SIZE
classificationDriver.CONTEXT_THRESHOLD = CONTEXT_THRESHOLD
classificationDriver.NUM_ITERATIONS = NUM_ITERATIONS
classificationDriver.MAX_RETRIES = MAX_RETRIES
classificationDriver.SILENT_MODE = SILENT_MODE
classificationDriver.CONFIDENCE_LEVEL = CONFIDENCE_LEVEL
classificationDriver.ERROR_MARGIN = ERROR_MARGIN
classificationDriver.ensureTmpFolder()

if len(pendingAppsDF) > 0:
    llmInterface = classificationDriver.initLlm(MODEL)

for appIdx, appRow in pendingAppsDF.iterrows():
    print("--- 🔑 Analyzing App [{}/{}] : {}".format(appIdx + 1, len(pendingAppsDF), appRow["sha256"]))
    appResult = classificationDriver.analyzeApkRecord(
        appRow.to_dict(), llmInterface, promptTemplate, EXPECTED_LABELS,
        obfuscationTechniqueDirName, numLabels=NUM_LABELS
    )
    classificationResults.append(appResult)
    AnalysisUtils.saveResults(classificationResults, OUTPUT_PATH_JSON)
    print("\n--- 💾 Partial Report saved!")
    print("\n" + "+++" * 20)


##### 💾 Save Results


In [ ]:
AnalysisUtils.saveResults(classificationResults, OUTPUT_PATH_JSON)
AnalysisUtils.saveResultsAsCsv(classificationResults, OUTPUT_PATH_CSV)
print("--- 💾 JSON Report saved : {}".format(OUTPUT_PATH_JSON))
print("--- 💾 CSV Report saved  : {}".format(OUTPUT_PATH_CSV))

#### 📊 Print some stats/results


In [ ]:
resultsDF = pd.DataFrame(classificationResults)
successDF = resultsDF[resultsDF["status"] == "SUCCESS"].copy() if len(resultsDF) > 0 else pd.DataFrame()

print("\n--- 📊 Dataset Statistics")
print("--- 🔹 Total Apps              : {}".format(len(resultsDF)))
print("--- 🔹 Success                 : {}".format(len(successDF)))
print("--- 🔹 Error                   : {}".format((resultsDF["status"].astype(str).str.startswith("ERROR -")).sum() if len(resultsDF) > 0 else 0))
print("--- 🔹 Zero Smali Classes      : {}".format((resultsDF["status"] == "NO_SMALI_CLASSES").sum() if len(resultsDF) > 0 else 0))
print("--- 🔹 No analyzable sampled classes: {}".format((resultsDF["status"] == "NO_ANALYZABLE_SMALI_CLASSES").sum() if len(resultsDF) > 0 else 0))

if NUM_LABELS == 3:
    print("--- 🔹 Three-label analysis only: accuracy requires ground-truth triples for each APK.")
    print("--- 🔹 The input folder identifies a group, not the three applied techniques.")
elif len(successDF) > 0:
    labelledSuccessDF = successDF[successDF["obfuscationTechnique"].isin(EXPECTED_LABELS)]
    if len(labelledSuccessDF) > 0:
        numCorrect = (labelledSuccessDF["llmFinalLabel"] == labelledSuccessDF["obfuscationTechnique"]).sum()
        print("--- 🔹 Exact-Match Accuracy    : {:.2%} ({} APKs with ground truth)".format(numCorrect / len(labelledSuccessDF), len(labelledSuccessDF)))
    else:
        print("--- 🔹 Accuracy unavailable: no recognized ground-truth technique labels.")
    print("--- 🔹 Predicted Label Counts  : {}".format(successDF["llmFinalLabel"].value_counts().to_dict()))

successDF.head(10)


##### 🔚 End


In [ ]:
endTime = datetime.datetime.now()
print("\n🔚 --- END:  {} --- 🔚".format(endTime.strftime("%Y-%m-%d %H:%M:%S")))

totalTime = endTime - initTime
hours = totalTime.total_seconds() // 3600
minutes = (totalTime.total_seconds() % 3600) // 60
print("⏱️ --- Time: {:02d} hours and {:02d} minutes [{:02d} seconds] --- ⏱️".format(int(hours), int(minutes), int(totalTime.total_seconds())))